# VibeVideo – Google Colab Setup

Run this notebook on a **T4 GPU** for fast object replacement.

> **Runtime → Change runtime type → T4 GPU**  ← Do this first!

---
Run each cell **in order** (▶ button). After setup is complete, use the **Command** cell at the bottom to type your editing commands.

## Step 1 — Clone the repo

In [ ]:
import os

REPO_DIR = "/content/VibeVideo"

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/ranchimall/VibeVideo.git {REPO_DIR}
else:
    print('Repo already cloned — pulling latest changes...')
    !git -C {REPO_DIR} pull



os.chdir(REPO_DIR)
print(f'Working directory: {os.getcwd()}')

print('Fetching ProPainter...')
!rm -rf ProPainter && git clone https://github.com/sczhou/ProPainter


## Step 2 — Install dependencies
This takes ~3-5 minutes on first run.

In [ ]:
# PyTorch with CUDA (Colab's T4 uses CUDA 12.x)
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121

# Core dependencies
!pip install -q \
    sentence-transformers \
    faiss-cpu \
    numpy \
    imageio-ffmpeg \
    insightface \
    onnxruntime-gpu \
    opencv-python-headless \
    yt-dlp \
    chromadb \
    easyocr \
    Pillow \
    openai-whisper \
    soundfile \
    ultralytics \
    lap \
    sam2

print('\nAll packages installed!')

## Step 3 — Download model weights
Downloads SAM2 Small (~856 MB). Only runs once per session.

In [ ]:
import os, urllib.request

os.makedirs('weights', exist_ok=True)

SAM2_URL  = 'https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_small.pt'
SAM2_DEST = 'weights/sam2.1_hiera_small.pt'

if os.path.exists(SAM2_DEST) and os.path.getsize(SAM2_DEST) > 850 * 1024 * 1024:
    print(f'SAM2 weights already present ({os.path.getsize(SAM2_DEST)/1024**2:.0f} MB)')
else:
    print('Downloading SAM2 Small weights (~856 MB)...')

    def _progress(count, block_size, total_size):
        pct = min(count * block_size / total_size * 100, 100)
        bar = '#' * int(pct // 2)
        print(f'\r  [{bar:<50}] {pct:5.1f}%', end='', flush=True)

    urllib.request.urlretrieve(SAM2_URL, SAM2_DEST, reporthook=_progress)
    print(f'\nDownloaded to {SAM2_DEST}')

## Step 4 — Verify GPU
Make sure you are on a T4 GPU before running commands.

In [ ]:
import torch

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    vram_gb = gpu.total_memory / 1024**3
    print(f'GPU: {gpu.name} ({vram_gb:.1f} GB VRAM)')
    if vram_gb >= 6:
        print('   SAM2 will run on GPU — fast mode enabled! ')
    else:
        print('   Warning: VRAM < 6 GB — SAM2 will fall back to CPU.')
else:
    print('No GPU detected! Go to Runtime → Change runtime type → T4 GPU')

## Step 5 — Upload your media files
Upload your videos and images to the `sample_media/` folder.

In [ ]:
from google.colab import files
import shutil, os

os.makedirs('sample_media', exist_ok=True)
print('Select your video/image files to upload:')
uploaded = files.upload()

for fname, data in uploaded.items():
    dest = os.path.join('sample_media', fname)
    with open(dest, 'wb') as f:
        f.write(data)
    print(f'  Saved: {dest}')

## Step 6 — Run VibeVideo

This starts the interactive CLI. Type your commands at the `Command:` prompt — for example:

```
replace the car in street_objects.mp4 with 1f697.png
```

When you are done, type `exit`.

> **Note:** Output files are saved in `sample_media/`. Download them in Step 7.

In [ ]:
%cd /content/VibeVideo
!python vibevideo.py

## Step 7 — Download your output

In [ ]:
from google.colab import files
import os

# Change this to the filename you want to download
OUTPUT_FILE = 'sample_media/replaced_output.mp4'

if os.path.exists(OUTPUT_FILE):
    files.download(OUTPUT_FILE)
else:
    print(f'File not found: {OUTPUT_FILE}')
    print('Available files in sample_media:')
    for f in sorted(os.listdir('sample_media')):
        print(f'  {f}')